# Модуль 3. Вебинар 3
## Новые модели и сравнение результатов

На прошлом вебинаре мы построили персональную модель на основе случайного леса. Сегодня сравним её с CatBoost, добавим признаки из истории чтения и проведём несколько контролируемых экспериментов.

## Что мы сделаем

1. Восстановим решение второго вебинара и его результат.
2. Подготовим категориальные признаки для CatBoost.
3. Обучим базовую модель CatBoost.
4. Добавим историю жанров и авторов.
5. Изменим один параметр модели и сравним результаты.
6. Выберем итоговое решение.
7. Переобучим модель на доступных данных и создадим финальный submission.

## Как работать с ученическим ноутбуком

В ноутбуке **18 самостоятельных заданий** и несколько итоговых проверок.  
Задания идут сразу после объяснения нового или важного шага.

## Главная логика вебинара

Каждый эксперимент должен отвечать на один вопрос:

- помогла ли новая модель;
- помог ли новый признак;
- помогло ли изменение параметра.

Если одновременно изменить всё, нельзя понять причину изменения метрики.

# 1. Загружаем данные

Используем тот же комплект участника, что и на предыдущих вебинарах.

In [ ]:
import os

data_folder = "stud"

# Проверяем, существует ли папка
if os.path.exists(data_folder):
    os.chdir(data_folder)
    print("Текущая папка:", os.getcwd())
else:
    print(f"Ошибка: папка '{data_folder}' не найдена")

Текущая папка: /Users/vsevolod/Desktop/Modul_3_Vebinar_3_Uchenicheskiy/stud


# 2. Устанавливаем и импортируем CatBoost

В Google Colab библиотеку CatBoost нужно установить перед первым импортом.

In [2]:
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier

from metric import ndcg_at_5 as pndcg_at_5
from metric import validate_submission

# 3. Читаем файлы

Структура данных не изменилась, поэтому используем знакомые команды.

In [3]:
users = pd.read_csv("users.csv")
books = pd.read_csv("books.csv")
candidates = pd.read_csv("candidates.csv")
valid_solution = pd.read_csv("valid_solution.csv")
sample_submission = pd.read_csv("sample_submission.csv")

interactions = pd.read_json(
    "interactions.jsonl",
    lines=True
)

train_choices = pd.read_json(
    "train_choices.jsonl",
    lines=True
)

# 4. Восстанавливаем обучающие пары

Как и на втором вебинаре:

- выбранные книги получают `target = 1`;
- остальные кандидаты получают `target = 0`.

In [4]:
# Задание 1.
# Подготовьте положительные пары:
# оставьте reader_id, period и book_id;
# переименуйте period в target_period;
# удалите повторы и создайте target = 1.

positive_pairs = (
    train_choices[
        ["reader_id", "period", "book_id"]
    ]
    .rename(columns={"period": "target_period"})
)
positive_pairs["target"] = 1

In [5]:
# Задание 2.
# Отберите train, valid и test.
# Затем объедините train_candidates с positive_pairs
# и заполните target значениями 0 и 1.

train_candidates = candidates[
    candidates["split"] == "train"
].copy()
valid_candidates = candidates[
    candidates["split"] == "valid"
].copy()
test_candidates = candidates[
    candidates["split"] == "test"
].copy()

train_table = train_candidates.merge(
    positive_pairs,
    on=["reader_id", "target_period", "book_id"],
    how="left"
)
train_table["target"] = (
    train_table["target"]
    .fillna(0)
    .astype(int)
)

# 5. Восстанавливаем базовые признаки

Сохраняем признаки второго вебинара и добавляем `age_match` – совпадение возрастной группы читателя и рекомендации книги.

In [6]:
# Задание 3.
# Рассчитайте reader_activity,
# book_popularity и book_avg_rating.

reader_activity = (
    interactions
    .groupby("reader_id")
    .size()
    .reset_index(name="reader_activity")
)
book_statistics = (
    interactions
    .groupby("book_id")
    .agg(
        book_popularity=("reader_id", "nunique"),
        book_avg_rating=("rating", "mean")
    )
    .reset_index()
)

In [7]:
# Задание 4.
# Допишите функцию базовых признаков.

def make_base_features(candidates_part):
    work = candidates_part.copy()

    work = work.merge(users, on="reader_id", how="left")
    work = work.merge(
        books,
        on="book_id",
        how="left",
        suffixes=("_reader", "_book")
    )

    work = work.merge(
        reader_activity,
        on="reader_id",
        how="left",
    )
    work = work.merge(
        book_statistics,
        on="book_id",
        how="left",
    )

    work["genre_match"] = (
        work["favorite_genre"] == work["genre"]
    ).astype(int)
    work["difficulty_match"] = (
        work["reading_level"] == work["difficulty"]
    ).astype(int)
    work["length_match"] = (
        work["preferred_length"] == work["length_group"]
    ).astype(int)
    work["age_match"] = (
        work["age_group_reader"] == work["age_group_book"]
    ).astype(int)

    numeric_columns = [
        "reader_activity",
        "book_popularity",
        "book_avg_rating"
    ]

    work[numeric_columns] = work[numeric_columns].fillna(0)
    return work

In [8]:
# Задание 5.
# Подготовьте базовые признаки для train и valid.

train_base = make_base_features(train_table)
valid_base = make_base_features(valid_candidates)

# 6. Функция формирования топ-5

Используем функцию предыдущего вебинара без изменений.

In [9]:
def make_submission_from_scores(table, score_column):
    work = table[
        ["reader_id", "book_id", score_column]
    ].copy()

    work = work.sort_values(
        ["reader_id", score_column, "book_id"],
        ascending=[True, False, True]
    )

    top_5 = work.groupby("reader_id").head(5)

    submission = (
        top_5
        .groupby("reader_id")["book_id"]
        .agg(" ".join)
        .reset_index()
        .rename(columns={"book_id": "recommendations"})
    )

    return submission

# 7. Точка отсчёта: RandomForest

Повторно считаем результат модели второго вебинара. Он нужен, чтобы оценить пользу CatBoost.

In [11]:
# Задание 6.
# Восстановите модель RandomForest второго вебинара,
# получите вероятности и посчитайте pNDCG@5.

random_forest_features = [
    "reader_activity",
    "book_popularity",
    "book_avg_rating",
    "genre_match",
    "difficulty_match",
    "length_match"
]
random_forest = RandomForestClassifier(
    n_estimators=100,
    max_depth=3,
    random_state=42
)
random_forest.fit(train_base[random_forest_features], train_base["target"])

valid_base["random_forest_score"] = random_forest.predict_proba(
    valid_base[random_forest_features]
)[:, 1]

random_forest_submission = make_submission_from_scores(
    valid_base, "random_forest_score")

random_forest_metric = pndcg_at_5(
    valid_solution, random_forest_submission, valid_candidates
)

print("pNDCG@5 RandomForest:", round(random_forest_metric, 6))

pNDCG@5 RandomForest: 0.582022


Ожидаемый результат – примерно **0.58**. Это наша точка отсчёта.

# 8. Чем CatBoost отличается от случайного леса

В таблице много категориальных признаков:

- жанр;
- автор;
- возрастная группа;
- уровень сложности;
- объём книги;
- идентификатор книги.

CatBoost умеет работать с такими столбцами напрямую. Нам не нужно вручную превращать каждую категорию в набор отдельных столбцов.

## `cat_features`

При обучении мы передаём CatBoost список названий категориальных признаков через параметр `cat_features`.

Категориальные значения приводим к строковому типу, чтобы модель однозначно воспринимала их как категории.

In [12]:
# Задание 7.
# Заполните списки числовых и категориальных признаков.

base_numeric_features = base_numeric_features = [
    "book_popularity",
    "book_avg_rating",
    "genre_match",
    "difficulty_match",
    "length_match",
    "age_match"
]

categorical_features = categorical_features = [
    "age_group_reader",
    "reading_level",
    "favorite_genre",
    "preferred_length",
    "author_id",
    "genre",
    "difficulty",
    "length_group",
    "age_group_book",
    "book_id"
]

catboost_base_features = base_numeric_features + categorical_features

In [13]:
# Задание 8.
# Преобразуйте категориальные столбцы train_base
# и valid_base в строковый тип.

for column in categorical_features:
    train_base[column] = train_base[column].astype(str)
    valid_base[column] = valid_base[column].astype(str)

# 9. Обучаем базовый CatBoost

Новые параметры:

- `iterations` – сколько последовательных деревьев строит модель;
- `learning_rate` – насколько сильно каждое новое дерево изменяет результат;
- `loss_function="Logloss"` – функция ошибки для бинарной классификации;
- `verbose=False` – не выводить процесс обучения после каждой итерации.

`depth` и `random_seed` имеют знакомый смысл: глубина деревьев и фиксированная случайность.

In [17]:
# Задание 9.
# Создайте базовый CatBoost с параметрами:
# iterations=200, depth=5, learning_rate=0.05,
# loss_function="Logloss", verbose=False, random_seed=42.
# Затем обучите модель.

catboost_base = CatBoostClassifier(
    iterations=200,
    depth=5,
    learning_rate=0.05,
    loss_function="Logloss",
    verbose=False,
    random_seed=42
)

catboost_base.fit(train_base[catboost_base_features], train_base["target"], cat_features=categorical_features)

CatBoostClassifier(depth=5, iterations=200, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=False)

In [18]:
# Задание 10.
# Получите вероятность класса 1,
# сформируйте submission и посчитайте метрику.

valid_base["catboost_base_score"] = catboost_base.predict_proba(
    valid_base[catboost_base_features]
)[:, 1]
catboost_base_submission = make_submission_from_scores(
    valid_base, "catboost_base_score")
catboost_base_metric = pndcg_at_5(
    valid_solution, catboost_base_submission, valid_candidates
)

print("pNDCG@5 CatBoost base:", round(catboost_base_metric, 6))

pNDCG@5 CatBoost base: 0.589115


При фиксированных параметрах результат должен быть примерно **0.60**. Изменили только модель и набор категориальных столбцов – результат стал немного выше.

# 10. Новая гипотеза: история авторов и жанров

Профиль читателя хранит только один любимый жанр. История взаимодействий даёт более подробную информацию.

Создадим два признака:

- `author_history_count` – сколько книг этого автора читатель выбирал раньше;
- `genre_history_count` – сколько книг этого жанра читатель выбирал раньше.

## Добавляем сведения о книгах в историю

В `interactions` есть `book_id`, но нет жанра и автора. Сначала присоединим эти столбцы из каталога книг.

In [19]:
# Задание 11.
# Добавьте к interactions жанр и author_id книги.

interactions_with_books = interactions.merge(
    books[["book_id", "genre", "author_id"]],
    on="book_id",
    how="left"
)
interactions_with_books.head()

,reader_id,period,book_id,rating,genre,author_id
0,r_00001,1,b_051,5,Научная фантастика,a_003
1,r_00001,1,b_075,4,Научная фантастика,a_017
2,r_00001,1,b_045,4,История,a_020
3,r_00001,1,b_073,3,Фэнтези,a_003
4,r_00001,2,b_059,4,Научная фантастика,a_001


## Считаем историю по автору

Группируем одновременно по читателю и автору. Одна строка результата отвечает на вопрос: сколько раз конкретный читатель выбирал книги конкретного автора.

In [22]:
# Задание 12.
# Для каждой пары reader_id + author_id
# посчитайте число прошлых взаимодействий.

author_history = interactions_with_books.groupby(
    ["reader_id", "author_id"]
).size().reset_index(name="author_history_count")
author_history.head(20)

,reader_id,author_id,author_history_count
0,r_00001,a_001,1
1,r_00001,a_003,2
2,r_00001,a_007,1
3,r_00001,a_009,1
4,r_00001,a_012,1
5,r_00001,a_017,1
6,r_00001,a_020,1
7,r_00002,a_003,1
8,r_00002,a_005,2
9,r_00002,a_009,2


## Считаем историю по жанру

Используем ту же логику, но группируем по `reader_id` и `genre`.

In [23]:
# Задание 13.
# Для каждой пары reader_id + genre
# посчитайте число прошлых взаимодействий.

genre_history = interactions_with_books.groupby(
    ["reader_id", "genre"]
).size().reset_index(name="genre_history_count")
genre_history.head()

,reader_id,genre,genre_history_count
0,r_00001,История,1
1,r_00001,Научная фантастика,3
2,r_00001,Психология,1
3,r_00001,Современная проза,1
4,r_00001,Фэнтези,2


# 11. Расширяем функцию подготовки признаков

После объединения с историей может появиться `NaN`: это означает, что читатель раньше не выбирал книги такого автора или жанра. Заменяем пропуск на `0`.

In [24]:
# Задание 14.
# Допишите функцию расширенных признаков.

def make_advanced_features(candidates_part):
    work = make_base_features(candidates_part)

    work = work.merge(
        author_history,
        on=["reader_id", "author_id"],
        how="left"
    )
    work = work.merge(
        genre_history,
        on=["reader_id", "genre"],
        how="left"
    )

    history_features = [
        "author_history_count",
        "genre_history_count"
    ]

    work[history_features] = work[history_features].fillna(0)
    return work

In [25]:
# Задание 15.
# Подготовьте train_advanced и valid_advanced,
# а затем преобразуйте категориальные столбцы в строки.

train_advanced = make_advanced_features(train_table)
valid_advanced = make_advanced_features(valid_candidates)

for column in categorical_features:
    train_advanced[column] = train_advanced[column].astype(str)
    valid_advanced[column] = valid_advanced[column].astype(str)

In [27]:
history_features = [
    "author_history_count",
    "genre_history_count"
]

advanced_features = (
    base_numeric_features
    + history_features
    + categorical_features
)

# 12. CatBoost с историческими признаками

Сохраняем параметры базовой модели. Меняем только признаки – так можно оценить пользу новой гипотезы.

In [28]:
# Задание 16.
# Обучите CatBoost с историческими признаками
# при тех же параметрах, что у базовой модели.

catboost_history = CatBoostClassifier(
    iterations=200,
    depth=5,
    learning_rate=0.05,
    loss_function="Logloss",
    verbose=False,
    random_seed=42
)
catboost_history.fit(train_advanced[advanced_features], train_advanced["target"], cat_features=categorical_features)

CatBoostClassifier(depth=5, iterations=200, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=False)

In [29]:
# Самостоятельная проверка.
# Получите прогноз, сформируйте top-5 и посчитайте метрику.

valid_advanced["catboost_history_score"] = catboost_history.predict_proba(
    valid_advanced[advanced_features])[:, 1]
catboost_history_submission = make_submission_from_scores(
    valid_advanced, "catboost_history_score")
catboost_history_metric = pndcg_at_5(
    valid_solution, catboost_history_submission, valid_candidates
)

print("pNDCG@5 CatBoost + history:", round(catboost_history_metric, 6))

pNDCG@5 CatBoost + history: 0.635334


Ожидаемый результат – примерно **0.64**. История авторов и жанров заметно усиливает персонализацию.

# 13. Фиксируем результаты экспериментов

In [30]:
experiment_results = pd.DataFrame(
    {
        "experiment": [
            "RandomForest",
            "CatBoost base",
            "CatBoost + history"
        ],
        "what_changed": [
            "точка отсчёта",
            "модель и категории",
            "добавлены история автора и жанра"
        ],
        "pNDCG@5": [
            random_forest_metric,
            catboost_base_metric,
            catboost_history_metric
        ]
    }
)

experiment_results

,experiment,what_changed,pNDCG@5
0,RandomForest,точка отсчёта,0.582022
1,CatBoost base,модель и категории,0.589115
2,CatBoost + history,добавлены история автора и жанра,0.635334


# 14. Эксперимент с параметром `depth`

Теперь признаки не меняем. Проверим только глубину деревьев: `3`, `5` и `7`.

Слишком маленькая глубина может не уловить связи, а слишком большая – сделать модель излишне сложной.

In [90]:
# Задание 17.
# Допишите функцию обучения и оценки модели
# для одного значения depth.

def train_catboost_with_depth(depth_value):
    model = CatBoostClassifier(
        iterations=200,
        depth=depth_value,
        learning_rate=0.05,
        loss_function="Logloss",
        verbose=False,
        random_seed=42
    )
    
    model.fit(train_advanced[advanced_features], train_advanced["target"], cat_features=categorical_features)

    probabilities = model.predict_proba(valid_advanced[advanced_features])[:, 1]

    temporary = valid_advanced[["reader_id", "book_id"]].copy()
    temporary["score"] = probabilities

    submission = make_submission_from_scores(temporary, "score")
    metric_value = pndcg_at_5(valid_solution, submission, valid_candidates)

    return model, metric_value

In [91]:
# Продолжение задания 17.
# Проверьте depth = 3, 5 и 7.

depth_results = []
depth_models = {}

for depth_value in [3, 5, 7]:
    model, metric_value = train_catboost_with_depth(depth_value)
    depth_models[depth_value] = model
    depth_results.append(
        {
            "depth": depth_value,
            "metric": metric_value
        }
    )

depth_results = pd.DataFrame(depth_results)
depth_results

,depth,metric
0,3,0.644620
1,5,0.635334
2,7,0.629833


Результаты могут немного отличаться в разных версиях библиотеки. Выбираем параметр по фактическому значению метрики, а не по ожиданию, что большая глубина обязательно лучше.

In [92]:
best_depth_row = (
    depth_results
    .sort_values("metric", ascending=False)
    .iloc[0]
)

best_depth = int(best_depth_row["depth"])
best_depth_metric = best_depth_row["metric"]

print("Лучшая глубина:", best_depth)
print("metric:", round(best_depth_metric, 6))

Лучшая глубина: 3
metric: 0.64462


# 15. Важность признаков итоговой модели

CatBoost, как и случайный лес, позволяет посмотреть относительную важность признаков. Это помогает сформулировать новые гипотезы, но не доказывает причинную связь.

In [93]:
best_validation_model = depth_models[best_depth]

feature_importance = pd.DataFrame(
    {
        "feature": advanced_features,
        "importance": best_validation_model.feature_importances_
    }
).sort_values(
    "importance",
    ascending=False
)

feature_importance.head(12)

,feature,importance
3,difficulty_match,32.825148
2,genre_match,15.844846
7,genre_history_count,14.458759
4,length_match,8.948906
6,author_history_count,8.199000
5,age_match,6.220328
0,book_popularity,6.206677
1,book_avg_rating,1.596237
14,difficulty,1.258610
9,reading_level,1.235866


# 16. Сравниваем рекомендации

Метрика показывает общий результат, а сравнение отдельных строк помогает увидеть, как изменился порядок книг.

In [94]:
recommendation_comparison = (
    random_forest_submission
    .merge(
        catboost_history_submission,
        on="reader_id",
        suffixes=("_random_forest", "_catboost")
    )
)

recommendation_comparison.head(10)

,reader_id,recommendations_random_forest,recommendations_catboost
0,r_00012,b_054 b_022 b_068 b_036 b_048,b_054 b_068 b_022 b_048 b_036
1,r_00015,b_050 b_010 b_034 b_054 b_004,b_010 b_050 b_014 b_054 b_034
2,r_00026,b_060 b_014 b_054 b_048 b_017,b_060 b_014 b_029 b_077 b_059
3,r_00030,b_025 b_057 b_027 b_075 b_028,b_057 b_025 b_027 b_075 b_008
4,r_00031,b_033 b_001 b_073 b_075 b_024,b_075 b_033 b_072 b_024 b_001
5,r_00034,b_031 b_015 b_069 b_023 b_058,b_069 b_031 b_015 b_058 b_016
6,r_00036,b_040 b_048 b_056 b_002 b_012,b_040 b_012 b_061 b_048 b_056
7,r_00037,b_020 b_028 b_076 b_036 b_035,b_028 b_076 b_020 b_036 b_026
8,r_00043,b_050 b_042 b_009 b_010 b_058,b_050 b_046 b_009 b_064 b_016
9,r_00046,b_038 b_070 b_022 b_073 b_030,b_070 b_042 b_038 b_030 b_073


# 17. Дополнительный финальный шаг: используем `train + valid`

Параметры уже выбраны. Теперь можно добавить открытые ответы `valid` к обучающей выборке и переобучить итоговую модель на большем количестве размеченных пар.

Для этого:

1. разделим строку рекомендаций на отдельные `book_id`;
2. создадим положительные пары для `valid`;
3. объединим размеченные `train` и `valid`.

## Новые операции: `str.split()`, `explode()` и `concat()`

- `.str.split()` превращает строку из пяти книг в список;
- `.explode()` превращает элементы списка в отдельные строки;
- `pd.concat()` соединяет таблицы друг под другом.

In [95]:
# Задание 18.
# Превратите valid_solution в положительные пары:
# разделите recommendations, примените explode(),
# добавьте target_period и target = 1.

valid_positive = valid_solution.copy()

valid_positive["book_id"] = valid_positive["recommendations"].str.split()
valid_positive = valid_positive.explode("book_id").drop(columns=["recommendations"])

valid_period = valid_candidates[["reader_id", "target_period"]].drop_duplicates()
valid_positive = valid_positive.merge(valid_period, on="reader_id", how="left")

valid_positive["target"] = 1


In [96]:
# Финальная самостоятельная часть.
# Создайте valid_labeled и объедините train + valid.

valid_labeled = valid_candidates.merge(
    valid_positive[["reader_id", "target_period", "book_id", "target"]],
    on=["reader_id", "target_period", "book_id"],
    how="left"
)
valid_labeled["target"] = valid_labeled["target"].fillna(0).astype(int)
train_and_valid = pd.concat([train_table, valid_labeled], ignore_index=True)

print("Строк train:", len(train_table))
print("Строк train + valid:", len(train_and_valid))

Строк train: 8000
Строк train + valid: 10000


# 18. Обучаем итоговую модель

Используем выбранную глубину и расширенные признаки. После добавления `valid` метрику на нём больше не считаем: эти ответы уже вошли в обучение.

In [87]:
# Финальная самостоятельная часть.
# Подготовьте признаки для train + valid и test.

final_train = make_advanced_features(train_and_valid)
test_advanced = make_advanced_features(test_candidates)

for column in categorical_features:
    final_train[column] = final_train[column].astype(str)
    test_advanced[column] = test_advanced[column].astype(str)

# 17.5. Проверка улучшений на валидации

Перед обучением финальной модели проверим, действительно ли новые параметры улучшают метрику на валидации.

In [76]:
# GRID SEARCH ПО ПАРАМЕТРАМ CATBOOST
# Перебираем все комбинации параметров для поиска лучшей конфигурации

import itertools

print("=" * 70)
print("GRID SEARCH ПО ПАРАМЕТРАМ")
print("=" * 70)

# Baseline для сравнения
baseline_metric = 0.644620
print(f"\nBaseline: {baseline_metric:.6f}")
print(f"Цель: превзойти baseline\n")

# Сетка параметров для перебора
param_grid = {
    'iterations': [150, 200, 250, 300, 400],
    'depth': [2, 3, 4, 5],
    'learning_rate': [0.03, 0.04, 0.05, 0.06, 0.07],
    'l2_leaf_reg': [1, 2, 3, 5]
}

print(f"Параметры для перебора:")
for param, values in param_grid.items():
    print(f"  {param}: {values}")

# Вычисляем общее количество комбинаций
total_combinations = 1
for values in param_grid.values():
    total_combinations *= len(values)
print(f"\nВсего комбинаций: {total_combinations}")
print(f"Это займет около {total_combinations * 3} сек (~{total_combinations * 3 / 60:.1f} мин)\n")

print("=" * 70)
print("НАЧИНАЕМ ПЕРЕБОР...")
print("=" * 70)

all_results = []
best_metric = baseline_metric
best_params = None
counter = 0

# Перебор всех комбинаций
for iterations in param_grid['iterations']:
    for depth in param_grid['depth']:
        for learning_rate in param_grid['learning_rate']:
            for l2_leaf_reg in param_grid['l2_leaf_reg']:
                counter += 1
                
                if counter % 20 == 0:
                    print(f"Прогресс: {counter}/{total_combinations} ({counter/total_combinations*100:.1f}%)")
                
                try:
                    model = CatBoostClassifier(
                        iterations=iterations,
                        depth=depth,
                        learning_rate=learning_rate,
                        l2_leaf_reg=l2_leaf_reg,
                        loss_function="Logloss",
                        verbose=False,
                        random_seed=42
                    )
                    
                    model.fit(
                        train_advanced[advanced_features], 
                        train_advanced["target"], 
                        cat_features=categorical_features
                    )
                    
                    valid_advanced["grid_score"] = model.predict_proba(
                        valid_advanced[advanced_features]
                    )[:, 1]
                    
                    temp_submission = make_submission_from_scores(valid_advanced, "grid_score")
                    metric = pndcg_at_5(valid_solution, temp_submission, valid_candidates)
                    
                    all_results.append({
                        'iterations': iterations,
                        'depth': depth,
                        'learning_rate': learning_rate,
                        'l2_leaf_reg': l2_leaf_reg,
                        'metric': metric
                    })
                    
                    # Обновляем лучший результат
                    if metric > best_metric:
                        best_metric = metric
                        best_params = {
                            'iterations': iterations,
                            'depth': depth,
                            'learning_rate': learning_rate,
                            'l2_leaf_reg': l2_leaf_reg
                        }
                        print(f"  🎯 НОВЫЙ ЛУЧШИЙ: {metric:.6f} (iter={iterations}, depth={depth}, lr={learning_rate}, l2={l2_leaf_reg})")
                
                except Exception as e:
                    print(f"Ошибка при комбинации: {e}")
                    continue

print("\n" + "=" * 70)
print("РЕЗУЛЬТАТЫ GRID SEARCH")
print("=" * 70)

# Топ-10 конфигураций
results_df = pd.DataFrame(all_results).sort_values('metric', ascending=False)
print("\nТОП-10 ЛУЧШИХ КОНФИГУРАЦИЙ:")
print(results_df.head(10).to_string(index=False))

print("\n" + "=" * 70)
if best_metric > baseline_metric:
    improvement = best_metric - baseline_metric
    print(f"✅ НАЙДЕНО УЛУЧШЕНИЕ!")
    print(f"\nЛучшая метрика: {best_metric:.6f} (+{improvement:.6f})")
    print(f"\nЛучшие параметры:")
    for param, value in best_params.items():
        print(f"  {param}: {value}")
else:
    print(f"⚠ УЛУЧШЕНИЯ НЕ НАЙДЕНО")
    print(f"Baseline остается лучшим: {baseline_metric:.6f}")
print("=" * 70)

GRID SEARCH ПО ПАРАМЕТРАМ

Baseline: 0.644620
Цель: превзойти baseline

Параметры для перебора:
  iterations: [150, 200, 250, 300, 400]
  depth: [2, 3, 4, 5]
  learning_rate: [0.03, 0.04, 0.05, 0.06, 0.07]
  l2_leaf_reg: [1, 2, 3, 5]

Всего комбинаций: 400
Это займет около 1200 сек (~20.0 мин)

НАЧИНАЕМ ПЕРЕБОР...
  🎯 НОВЫЙ ЛУЧШИЙ: 0.645305 (iter=150, depth=2, lr=0.06, l2=1)
Прогресс: 20/400 (5.0%)
Прогресс: 40/400 (10.0%)
  🎯 НОВЫЙ ЛУЧШИЙ: 0.645542 (iter=150, depth=4, lr=0.06, l2=5)
Прогресс: 60/400 (15.0%)
Прогресс: 80/400 (20.0%)
  🎯 НОВЫЙ ЛУЧШИЙ: 0.647777 (iter=150, depth=5, lr=0.07, l2=5)
Прогресс: 100/400 (25.0%)
  🎯 НОВЫЙ ЛУЧШИЙ: 0.648109 (iter=200, depth=3, lr=0.03, l2=5)
Прогресс: 120/400 (30.0%)
Прогресс: 140/400 (35.0%)
Прогресс: 160/400 (40.0%)
Прогресс: 180/400 (45.0%)
Прогресс: 200/400 (50.0%)
Прогресс: 220/400 (55.0%)
Прогресс: 240/400 (60.0%)
Прогресс: 260/400 (65.0%)
Прогресс: 280/400 (70.0%)
Прогресс: 300/400 (75.0%)
Прогресс: 320/400 (80.0%)
Прогресс: 340/400 (85.0%)

In [97]:
# Финальная самостоятельная часть.
# Создайте и обучите итоговую модель
# с выбранным best_depth.

# Используем BASELINE параметры - они показали лучший результат на валидации!
# Тестирование подтвердило: depth=3, iterations=200, lr=0.05 - оптимальны

final_model = CatBoostClassifier(
    iterations=200,
    depth=best_depth,  # = 3
    learning_rate=0.03,
    l2_leaf_reg=5,
    loss_function="Logloss",
    verbose=False,
    random_seed=42
)
final_model.fit(
    final_train[advanced_features], 
    final_train["target"], 
    cat_features=categorical_features
)

CatBoostClassifier(depth=3, iterations=200, l2_leaf_reg=5, learning_rate=0.03, loss_function='Logloss', random_seed=42, verbose=False)

In [98]:
# Финальная самостоятельная часть.
# Получите прогноз, сформируйте submission,
# проверьте формат и сохраните CSV.

test_advanced["final_score"] = final_model.predict_proba(
    test_advanced[advanced_features])[:, 1]
final_submission = make_submission_from_scores(test_advanced, "final_score")

validate_submission(
    final_submission,
    candidates=test_candidates,
    expected_readers=sample_submission["reader_id"]
)

final_submission.to_csv(
    "submission_catboost.csv",
    index=False
)

final_submission.head()

,reader_id,recommendations
0,r_00004,b_031 b_075 b_038 b_003 b_051
1,r_00008,b_067 b_043 b_011 b_060 b_057
2,r_00013,b_071 b_012 b_072 b_041 b_011
3,r_00014,b_003 b_059 b_051 b_043 b_071
4,r_00018,b_075 b_001 b_067 b_041 b_036


# 19. Итоговая таблица экспериментов

Перед завершением добавим эксперимент с выбранной глубиной в журнал результатов.

In [99]:
final_experiment_row = pd.DataFrame(
    {
        "experiment": ["CatBoost best depth"],
        "what_changed": [
            "изменён только параметр depth"
        ],
        "pNDCG@5": [best_depth_metric]
    }
)

experiment_results = pd.concat(
    [experiment_results, final_experiment_row],
    ignore_index=True
)

experiment_results.sort_values(
    "pNDCG@5",
    ascending=False
)

,experiment,what_changed,pNDCG@5
3,CatBoost best depth,изменён только параметр depth,0.644620
4,CatBoost best depth,изменён только параметр depth,0.644620
5,CatBoost best depth,изменён только параметр depth,0.644620
6,CatBoost best depth,изменён только параметр depth,0.644620
7,CatBoost best depth,изменён только параметр depth,0.644620
2,CatBoost + history,добавлены история автора и жанра,0.635334
1,CatBoost base,модель и категории,0.589115
0,RandomForest,точка отсчёта,0.582022


# 20. Итоги вебинара

Сегодня мы:

- сравнили RandomForest и CatBoost;
- передали категориальные признаки через `cat_features`;
- добавили историю авторов и жанров;
- проверили гипотезу по изменению метрики;
- сравнили несколько значений `depth`;
- выбрали итоговые параметры;
- переобучили модель на `train + valid`;
- создали финальный submission.

Главный результат – не конкретное число метрики, а последовательный процесс: **гипотеза → одно изменение → проверка → вывод**.

## Самостоятельное продолжение

Можно проверить ещё одну гипотезу, изменяя только один элемент:

- оставить только `author_history_count`;
- оставить только `genre_history_count`;
- сравнить `iterations=100`, `200` и `400`;
- добавить совпадение автора с самым часто читаемым автором;
- посмотреть, меняется ли топ важных признаков.